# Chart Transport Writeup

Train smoke-scale stochastic and deterministic chart-transport runs for the 3D-ambient / 2D-support Gaussian-mixture setup, save dense checkpoints, then render checkpoint-conditioned writeup resources.

In [ ]:
from pathlib import Path

import torch

from src.data.gaussian_mixture.data import MultimodalGaussianDataConfig
from src.deterministic_chart_transport.constraint import (
    ChartPretrainConfig as DeterministicChartPretrainConfig,
)
from src.deterministic_chart_transport.constraint import (
    IntegratedChartConstraintConfig as DeterministicIntegratedChartConstraintConfig,
)
from src.deterministic_chart_transport.constraint import (
    LatentNormAnchorConfig as DeterministicLatentNormAnchorConfig,
)
from src.deterministic_chart_transport.constraint import (
    LatentScaleAnchorConfig as DeterministicLatentScaleAnchorConfig,
)
from src.deterministic_chart_transport.constraint import (
    ReconstructionConfig as DeterministicReconstructionConfig,
)
from src.deterministic_chart_transport.critic import (
    CriticLossConfig as DeterministicCriticLossConfig,
)
from src.deterministic_chart_transport.model import ChartTransportModelConfig as DeterministicChartTransportModelConfig
from src.deterministic_chart_transport.study import DeterministicChartTransportStudyConfig
from src.deterministic_chart_transport.transport import DeterministicChartTransportLossConfig
from src.model.mlp import StackedResidualMLPConfig
from src.model.time_conditioning import (
    CategoricalConditioningConfig,
    TimeConditioningConfig,
)
from src.priors.anchored import AnchoredGaussianScaleMixturePriorConfig
from src.stochastic_chart_transport.constraint import (
    ChartPretrainConfig as StochasticChartPretrainConfig,
)
from src.stochastic_chart_transport.constraint import (
    IntegratedChartConstraintConfig as StochasticIntegratedChartConstraintConfig,
)
from src.stochastic_chart_transport.constraint import (
    LatentNormAnchorConfig as StochasticLatentNormAnchorConfig,
)
from src.stochastic_chart_transport.constraint import (
    LatentScaleAnchorConfig as StochasticLatentScaleAnchorConfig,
)
from src.stochastic_chart_transport.constraint import (
    ReconstructionConfig as StochasticReconstructionConfig,
)
from src.stochastic_chart_transport.critic import (
    CriticLossConfig as StochasticCriticLossConfig,
)
from src.stochastic_chart_transport.fibers import FlatFiberPacking
from src.stochastic_chart_transport.model import ChartTransportModelConfig as StochasticChartTransportModelConfig
from src.stochastic_chart_transport.study import StochasticChartTransportStudyConfig
from src.stochastic_chart_transport.transport import StochasticChartTransportLossConfig
from src.writeup_helpers.chart_transport_writeup import (
    ChartTransportResourceConfig,
    ChartTransportTrainingPhaseConfig,
    ChartTransportTwoPanelFigureConfig,
    CheckpointSelectionConfig,
    DeterministicChartTransportWriteupConfig,
    IntegratedTrainingPhaseConfig,
    StochasticChartTransportWriteupConfig,
    render_deterministic_writeup_resources,
    render_stochastic_writeup_resources,
    run_deterministic_writeup_training,
    run_stochastic_writeup_training,
)

torch.manual_seed(0)
preferred_device_name = 'cuda:0'

## Shared Resource Config

In [ ]:
resource_config = ChartTransportResourceConfig(
    visualize_batch_size_per_mode=256,
    model_latent_batch_size=2_048,
    checkpoint_selection=CheckpointSelectionConfig(
        start_step=100,
        end_step=1_000,
        step_stride=100,
    ),
    figure=ChartTransportTwoPanelFigureConfig(
        marker_size=3.0,
        marker_opacity=0.55,
        width=1_200,
        height=520,
        combined_width=3_200,
        combined_height=1_150,
        camera_eye=(1.6, 1.55, 1.15),
    ),
)
display(resource_config.visualize())

## Stochastic

In [ ]:
stochastic_artifact_root = Path('artifacts/writeup/stochastic_chart_transport_3d_ambient')
num_modes = 8
batch_size = 2_048
ambient_dimension = 3
latent_dimension = ambient_dimension
fiber_dimension = 1
hidden_dim = 512
hidden_dims = [hidden_dim] * 4

stochastic_model_config = StochasticChartTransportModelConfig(
    encoder=StackedResidualMLPConfig.initialize(
        layer_dims=[ambient_dimension + fiber_dimension] + hidden_dims + [latent_dimension]
    ),
    decoder=StackedResidualMLPConfig.initialize(
        layer_dims=[latent_dimension] + hidden_dims + [ambient_dimension + fiber_dimension]
    ),
    critic=StackedResidualMLPConfig.initialize(
        layer_dims=[latent_dimension] + hidden_dims + [latent_dimension],
        time_conditioning_config=TimeConditioningConfig(
            min_t_lambda=1e-3,
            max_t_lambda=1.0,
            condition_dim=hidden_dim,
        ),
        cat_conditioning_config=CategoricalConditioningConfig(
            num_classes=2,
            condition_dim=hidden_dim,
        ),
    ),
    chart_lr=3e-4,
    critic_lr=3e-4,
    grad_clip_norm=2.0,
)
stochastic_data_reconstruction = StochasticReconstructionConfig(
    huber_delta=10.0,
    weight=50.0,
)
stochastic_integrated_anchor = StochasticLatentScaleAnchorConfig(
    latent_norm_weight=1.0,
    latent_zero_mean_weight=1.0,
    target_norm_per_dimension=1.0,
)
stochastic_run_config = StochasticChartTransportWriteupConfig(
    artifact_root=stochastic_artifact_root,
    seed=0,
    device_name=preferred_device_name,
    batch_size=batch_size,
    pretrain=ChartTransportTrainingPhaseConfig(num_steps=1_000),
    critic=ChartTransportTrainingPhaseConfig(num_steps=1_000),
    integrated=IntegratedTrainingPhaseConfig(
        num_steps=1_000,
        checkpoint_every_n_steps=1,
        transport_chart_every_n_steps=5,
    ),
    resources=resource_config,
    study=StochasticChartTransportStudyConfig(
        data=MultimodalGaussianDataConfig.initialize(
            num_modes=num_modes,
            mode_radius=1.0,
            mode_std=0.05,
            ambient_dimension=ambient_dimension,
            offset=0.0,
        ),
        prior=AnchoredGaussianScaleMixturePriorConfig(
            latent_shape=[latent_dimension],
            precision=4.0,
        ),
        model=stochastic_model_config,
        pretrain=StochasticChartPretrainConfig(
            data_reconstruction=stochastic_data_reconstruction,
            model_reconstruction=StochasticReconstructionConfig(
                huber_delta=10.0,
                weight=0.0,
            ),
            data_fiber_reconstruction=StochasticReconstructionConfig(
                huber_delta=10.0,
                weight=0.1,
            ),
            prior_reconstruction=StochasticReconstructionConfig(
                huber_delta=10.0,
                weight=1.0,
            ),
            anchor_config=StochasticLatentNormAnchorConfig(
                latent_norm_weight=1e-3,
            ),
        ),
        critic=StochasticCriticLossConfig(
            huber_delta=10.0,
            weight=1.0,
            t_min=0.005,
            t_max=0.99,
        ),
        transport=StochasticChartTransportLossConfig(
            transport_step_multiplier=0.05,
            transport_step_cap=0.05,
            num_time_samples=5,
            t_range=(0.01, 0.99),
            antipodal_estimate=True,
            decoder_huber_delta=5.0,
            forward_kl_weight=10.0,
            reverse_kl_weight=10.0,
            data_decoder_weight_multiplier=1.0,
        ),
        integrated_constraint=StochasticIntegratedChartConstraintConfig(
            data_reconstruction=stochastic_data_reconstruction,
            model_reconstruction=StochasticReconstructionConfig(
                huber_delta=10.0,
                weight=0.0,
            ),
            data_latent_anchor=stochastic_integrated_anchor,
            model_latent_anchor=stochastic_integrated_anchor,
        ),
        fiber_packing=FlatFiberPacking(fiber_ndims=fiber_dimension),
    ),
)
display(stochastic_run_config.visualize())

In [ ]:
stochastic_training = run_stochastic_writeup_training(
    config=stochastic_run_config,
)
{
    key: len(value)
    for key, value in stochastic_training.history.items()
}

In [ ]:
stochastic_render = render_stochastic_writeup_resources(
    config=stochastic_run_config,
)
stochastic_render.combined_figure

## Deterministic

In [ ]:
deterministic_artifact_root = Path('artifacts/writeup/deterministic_chart_transport_3d_ambient')

deterministic_model_config = DeterministicChartTransportModelConfig(
    encoder=StackedResidualMLPConfig.initialize(
        layer_dims=[ambient_dimension] + hidden_dims + [latent_dimension]
    ),
    decoder=StackedResidualMLPConfig.initialize(
        layer_dims=[latent_dimension] + hidden_dims + [ambient_dimension]
    ),
    critic=StackedResidualMLPConfig.initialize(
        layer_dims=[latent_dimension] + hidden_dims + [latent_dimension],
        time_conditioning_config=TimeConditioningConfig(
            min_t_lambda=1e-3,
            max_t_lambda=1.0,
            condition_dim=hidden_dim,
        ),
    ),
    chart_lr=3e-4,
    critic_lr=3e-4,
    grad_clip_norm=2.0,
)
deterministic_data_reconstruction = DeterministicReconstructionConfig(
    huber_delta=10.0,
    weight=50.0,
)
deterministic_run_config = DeterministicChartTransportWriteupConfig(
    artifact_root=deterministic_artifact_root,
    seed=0,
    device_name=preferred_device_name,
    batch_size=batch_size,
    pretrain=ChartTransportTrainingPhaseConfig(num_steps=1_000),
    critic=ChartTransportTrainingPhaseConfig(num_steps=1_000),
    integrated=IntegratedTrainingPhaseConfig(
        num_steps=1_000,
        checkpoint_every_n_steps=1,
        transport_chart_every_n_steps=5,
    ),
    resources=resource_config,
    study=DeterministicChartTransportStudyConfig(
        data=MultimodalGaussianDataConfig.initialize(
            num_modes=num_modes,
            mode_radius=1.0,
            mode_std=0.05,
            ambient_dimension=ambient_dimension,
            offset=0.0,
        ),
        prior=AnchoredGaussianScaleMixturePriorConfig(
            latent_shape=[latent_dimension],
            precision=4.0,
        ),
        model=deterministic_model_config,
        pretrain=DeterministicChartPretrainConfig(
            data_reconstruction=deterministic_data_reconstruction,
            prior_roundtrip=DeterministicReconstructionConfig(
                huber_delta=10.0,
                weight=1.0,
            ),
            anchor_config=DeterministicLatentNormAnchorConfig(
                latent_norm_weight=1e-3,
            ),
        ),
        critic=DeterministicCriticLossConfig(
            huber_delta=10.0,
            weight=1.0,
            t_min=0.005,
            t_max=0.99,
        ),
        transport=DeterministicChartTransportLossConfig(
            transport_step_multiplier=0.05,
            transport_step_cap=0.05,
            num_time_samples=5,
            t_range=(0.01, 0.99),
            antipodal_estimate=True,
            decoder_huber_delta=5.0,
            encoder_transport_weight=10.0,
            decoder_transport_weight=10.0,
        ),
        integrated_constraint=DeterministicIntegratedChartConstraintConfig(
            data_reconstruction=deterministic_data_reconstruction,
            prior_roundtrip=DeterministicReconstructionConfig(
                huber_delta=10.0,
                weight=1.0,
            ),
            data_latent_anchor=DeterministicLatentScaleAnchorConfig(
                latent_norm_weight=1.0,
                latent_zero_mean_weight=1.0,
                target_norm_per_dimension=1.0,
            ),
        ),
    ),
)
display(deterministic_run_config.visualize())

In [ ]:
deterministic_training = run_deterministic_writeup_training(
    config=deterministic_run_config,
)
{
    key: len(value)
    for key, value in deterministic_training.history.items()
}

In [ ]:
deterministic_render = render_deterministic_writeup_resources(
    config=deterministic_run_config,
)
deterministic_render.combined_figure